In [ ]:
# =======================================================
# COLAB B: PERSISTENT ENVIRONMENT & TAILSCALE SETUP
# =======================================================
import os
import time
from google.colab import drive

# --- STEP 0: MOUNT GOOGLE DRIVE FOR NODE PERSISTENCE ---
print("📁 Mounting Google Drive to persist Tailscale identity...")
drive.mount('/content/drive')

# Create a dedicated folder on Drive for Colab B's state
DRIVE_STATE_DIR = "/content/drive/MyDrive/Colab_B_Tailscale_State"
os.makedirs(DRIVE_STATE_DIR, exist_ok=True)
DRIVE_STATE_FILE = os.path.join(DRIVE_STATE_DIR, "tailscaled.state")

# --- STEP 1: PREVENT APT LOCKS & CORE DEPENDENCIES ---
print("Initializing environment and clearing package locks...")
os.system("sudo rm -f /var/lib/dpkg/lock-frontend /var/lib/apt/lists/lock >/dev/null 2>&1")
os.system("sudo apt-get update -y >/dev/null 2>&1")

# --- STEP 2: INSTALL GRAPHICS & TYPOGRAPHY FONTS ---
print("Installing Core Windows & Apple Fonts, and Mesa software rendering stack...")
os.system("echo 'ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true' | sudo debconf-set-selections")
os.system("sudo apt-get install -y xvfb xdotool ca-certificates curl gnupg libgl1-mesa-dri mesa-utils libegl1-mesa ttf-mscorefonts-installer fonts-liberation fonts-noto-color-emoji fonts-roboto >/dev/null 2>&1")
os.system("sudo fc-cache -f -v >/dev/null 2>&1")

# --- STEP 3: INSTALL OFFICIAL GOOGLE CHROME ---
print("Installing official Google Chrome...")
os.system("wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | sudo gpg --yes --dearmor -o /usr/share/keyrings/googlechrome-keyring.gpg")
os.system("echo 'deb [arch=amd64 signed-by=/usr/share/keyrings/googlechrome-keyring.gpg] http://dl.google.com/linux/chrome/deb/ stable main' | sudo tee /etc/apt/sources.list.d/google-chrome.list >/dev/null")
os.system("sudo apt-get update -y >/dev/null 2>&1")
os.system("sudo apt-get install -y google-chrome-stable >/dev/null 2>&1")

# --- STEP 4: INSTALL PYTHON STEALTH LIBRARIES ---
print("Installing Python packages...")
os.system("pip install undetected-chromedriver pyautogui PySocks beautifulsoup4 requests >/dev/null 2>&1")

# --- STEP 5: INSTALL TAILSCALE ---
print("Installing Tailscale...")
os.system("sudo mkdir -p /usr/share/keyrings >/dev/null 2>&1")
os.system("curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg | sudo tee /usr/share/keyrings/tailscale-archive-keyring.gpg >/dev/null")
os.system("curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list | sudo tee /etc/apt/sources.list.d/tailscale.list >/dev/null")
os.system("sudo apt-get update -y >/dev/null 2>&1")
os.system("sudo apt-get install -y tailscale >/dev/null 2>&1")

# --- STEP 6: RESTORE PERSISTENT STATE ---
print("🔄 Resetting sockets and preparing Persistent State...")
os.system("sudo pkill -9 tailscaled >/dev/null 2>&1")
os.system("sudo rm -f /var/run/tailscale/tailscaled.sock >/dev/null 2>&1")

TS_STATE_DIR = "/var/lib/tailscale"
os.system(f"sudo mkdir -p {TS_STATE_DIR}")

# If the state file exists on Drive, copy it to the local system BEFORE starting the daemon
if os.path.exists(DRIVE_STATE_FILE):
    print("🔄 Found existing Tailscale identity for Colab B in Drive! Restoring...")
    os.system(f"sudo cp {DRIVE_STATE_FILE} {TS_STATE_DIR}/tailscaled.state")
else:
    print("🆕 No existing identity found. A new one will be created.")

# --- STEP 7: BOOT DAEMON & AUTHENTICATE ---
print("Booting Tailscale daemon...")
os.system("nohup sudo tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > tailscaled.log 2>&1 &")
time.sleep(3)

print("🔗 Authenticating Tailscale...")
# If state was restored, this will instantly pass. Otherwise, it will print the login URL.
!sudo tailscale up

# --- STEP 8: SAVE NEW STATE ---
print("💾 Saving Tailscale identity back to Google Drive...")
os.system(f"sudo cp {TS_STATE_DIR}/tailscaled.state {DRIVE_STATE_FILE}")

print("✅ Initialization Complete! Colab B is now a persistent node.")

📁 Mounting Google Drive to persist Tailscale identity...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Initializing environment and clearing package locks...
Installing Core Windows & Apple Fonts, and Mesa software rendering stack...
Installing official Google Chrome...
Installing Python packages...
Installing Tailscale...
🔄 Resetting sockets and preparing Persistent State...
🆕 No existing identity found. A new one will be created.
Booting Tailscale daemon...
🔗 Authenticating Tailscale...

To authenticate, visit:

	https://login.tailscale.com/a/7ba20f3014a7b

Success.
💾 Saving Tailscale identity back to Google Drive...
✅ Initialization Complete! Colab B is now a persistent node.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- CELL 2: UNIVERSAL STEALTH SCRAPER & TRANSMITTER ---
# @title Configure Networking & Master Node
EXIT_NODE_IP = "100.86.149.127" #@param {type:"string"}
COLAB_A_IP = "100.96.27.123"    #@param {type:"string"}

# ====================================================================
# --- 1. STANDARD IMPORTS ---
# ====================================================================
import os
import json
import time
import math
import shutil
import subprocess
import socket
import socks
import requests

# ====================================================================
# --- 2. PREVENT XAUTH ERROR IMMEDIATELY ---
# ====================================================================
xauth_path = os.path.expanduser("~/.Xauthority")
try:
    if not os.path.exists(xauth_path):
        with open(xauth_path, "wb") as f:
            f.write(b"")
except Exception:
    pass

# ====================================================================
# --- 3. SELF-HEALING TAILSCALE DAEMON ---
# ====================================================================
def ensure_tailscale_running():
    try:
        subprocess.check_output(["tailscale", "status"], stderr=subprocess.STDOUT)
    except subprocess.CalledProcessError:
        print("⚠️ Colab Tailscale daemon is offline! Rebooting it now...")
        os.system("sudo pkill -9 tailscaled")
        os.system("sudo rm -f /var/run/tailscale/tailscaled.sock")
        os.system("nohup sudo tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > tailscaled.log 2>&1 &")
        time.sleep(4)

ensure_tailscale_running()

print(f"🔒 Locking Tailscale routing strictly to Exit Node: {EXIT_NODE_IP}...")
os.system(f"sudo tailscale set --exit-node={EXIT_NODE_IP} > /dev/null 2>&1")
time.sleep(3)

# ====================================================================
# --- 4. CLEAN PERSISTENT CACHE TO PREVENT SERVICE WORKER LEAKS ---
# ====================================================================
user_data_dir = os.path.abspath("chrome_stealth_profile")

def purge_cached_workers(profile_dir):
    paths_to_delete = [
        os.path.join(profile_dir, "Default", "Service Worker"),
        os.path.join(profile_dir, "Default", "Cache"),
        os.path.join(profile_dir, "Default", "Code Cache")
    ]
    for path in paths_to_delete:
        if os.path.exists(path):
            try:
                shutil.rmtree(path)
            except Exception:
                pass

print("🧹 Cleaning persistent Service Worker cache...")
purge_cached_workers(user_data_dir)

# ====================================================================
# --- 5. DYNAMIC SYSTEM ENGINE (UNIVERSAL MAPPER) ---
# ====================================================================
def get_dynamic_profile(ip):
    cv_str = subprocess.check_output(['google-chrome', '--version']).decode('utf-8').strip().split()[2]
    major_v = cv_str.split('.')[0]

    try:
        status_json_str = subprocess.check_output(["tailscale", "status", "--json"]).decode("utf-8")
        status_data = json.loads(status_json_str)
        peers = status_data.get("Peer", {})

        target_peer = next((p for p in peers.values() if ip in p.get("TailscaleIPs", [])), None)

        os_type = target_peer.get("OS", "").lower() if target_peer else ""
        arch_type = target_peer.get("GoArch", "").lower() if target_peer else ""

        if target_peer and "Hostinfo" in target_peer:
            if not os_type:
                os_type = target_peer["Hostinfo"].get("OS", "").lower()
            if not arch_type:
                arch_type = target_peer["Hostinfo"].get("GoArch", "").lower()

        print(f"✅ Dynamic Engine detected Exit Node: OS='{os_type.upper()}' | ARCH='{arch_type.upper()}'")
    except Exception as e:
        print(f"⚠️ Failed to query Tailscale. Defaulting to Linux x86_64 profile. Error: {e}")
        os_type, arch_type = "linux", "amd64"

    # Default Base Profile (Linux Desktop)
    profile = {
        "width": 1920,
        "height": 1080,
        "is_mobile": False,
        "cv_full": cv_str,
        "cv_major": major_v,
        "cores": 8,
        "ram": 16,
        "battery_charging": "true",
        "battery_level": "1.0",
        "cam_name": "HD Web Camera",
        "platform": "Linux x86_64",
        "ch_platform": "Linux",
        "ch_arch": "x86",
        "ch_version": "5.15.0",
        "user_agent": f"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/{cv_str} Safari/537.36",
        "webgl_vendor": "Google Inc. (NVIDIA)",
        "webgl_renderer": "ANGLE (NVIDIA, NVIDIA GeForce GTX 1050 Direct3D11 vs_5_0 ps_5_0, D3D11)",
        "timezone": "Asia/Kolkata"
    }

    if "mac" in os_type and "arm" in arch_type:
        profile.update({
            "platform": "MacIntel", "ch_platform": "macOS", "ch_arch": "arm", "ch_version": "10.15.7",
            "cores": 8, "ram": 8, "cam_name": "FaceTime HD Camera",
            "user_agent": f"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/{cv_str} Safari/537.36",
            "webgl_vendor": "Apple Inc.", "webgl_renderer": "Apple M1"
        })
    elif "mac" in os_type:
        profile.update({
            "platform": "MacIntel", "ch_platform": "macOS", "ch_arch": "x86", "ch_version": "10.15.7",
            "cores": 8, "ram": 16, "cam_name": "FaceTime HD Camera",
            "user_agent": f"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/{cv_str} Safari/537.36",
            "webgl_vendor": "Apple Inc.", "webgl_renderer": "Intel(R) Iris(TM) Plus Graphics 640"
        })
    elif "win" in os_type:
        profile.update({
            "platform": "Win32", "ch_platform": "Windows", "ch_arch": "x86", "ch_version": "10.0.0",
            "cores": 16, "ram": 16, "cam_name": "Integrated Camera",
            "user_agent": f"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/{cv_str} Safari/537.36",
            "webgl_vendor": "Google Inc. (NVIDIA)", "webgl_renderer": "ANGLE (NVIDIA, NVIDIA GeForce RTX 3060 Direct3D11 vs_5_0 ps_5_0, D3D11)"
        })
    elif "android" in os_type or "ios" in os_type:
        print("📱 Morphing browser into Mobile Smartphone profile...")
        profile.update({
            "width": 360, "height": 800, "is_mobile": True, "cores": 8, "ram": 8,
            "battery_charging": "false", "battery_level": "0.85", "cam_name": "Front Camera",
            "platform": "Linux armv8l", "ch_platform": "Android", "ch_arch": "arm", "ch_version": "13.0.0",
            "user_agent": f"Mozilla/5.0 (Linux; Android 13; SM-S901B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/{cv_str} Mobile Safari/537.36",
            "webgl_vendor": "Qualcomm", "webgl_renderer": "Adreno (TM) 730"
        })

    return profile

sys_info = get_dynamic_profile(EXIT_NODE_IP)

# ====================================================================
# --- 6. SOCKS5 DIAGNOSTIC & DYNAMIC GEOLOCATION FETCHER ---
# ====================================================================
print("\n📡 Checking SOCKS5 connectivity & fetching dynamic exit node geolocation...")
while True:
    try:
        test_socket = socks.socksocket()
        test_socket.set_proxy(socks.SOCKS5, "127.0.0.1", 1055)
        test_socket.settimeout(6)
        test_socket.connect(("1.1.1.1", 80))
        test_socket.close()

        proxies = {'http': 'socks5h://127.0.0.1:1055', 'https': 'socks5h://127.0.0.1:1055'}
        geo_data = requests.get("http://ip-api.com/json/", proxies=proxies, timeout=10).json()
        dynamic_timezone = geo_data.get("timezone", "UTC")

        print(f"✅ SOCKS5 proxy is healthy!")
        print(f"🌍 Exit Node Location: {geo_data.get('city', 'Unknown')}, {geo_data.get('country', 'Unknown')} (Timezone: {dynamic_timezone})\n")

        sys_info["timezone"] = dynamic_timezone
        break
    except Exception as e:
        print(f"\n❌ SOCKS5 PROXY UNREACHABLE! Please wake up your target device ({EXIT_NODE_IP}). Retrying in 10s...")
        time.sleep(10)

# ====================================================================
# --- 7. SETUP VIRTUAL DISPLAY (STABLE GEOMETRY) ---
# ====================================================================
print("🖥️ Starting Xvfb Virtual Display (1920x1080)...")
os.system("pkill -9 -f Xvfb")
os.system("pkill -9 Xvfb")
os.system("rm -f /tmp/.X99-lock")
os.system("Xvfb :99 -screen 0 1920x1080x24 > /dev/null 2>&1 &")
os.environ['DISPLAY'] = ':99'
time.sleep(2)

# ====================================================================
# --- 8. DEFERRED IMPORTS (STRICTLY AFTER DISPLAY BOOT) ---
# ====================================================================
import pyautogui
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

pyautogui.FAILSAFE = False

# ====================================================================
# --- 9. CONFIGURE CHROME ---
# ====================================================================
options = uc.ChromeOptions()
options.add_argument('--proxy-server=socks5://127.0.0.1:1055')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# Swiftshader software rendering overrides (Fixes missing WebGL Context)
options.add_argument('--use-gl=angle')
options.add_argument('--use-angle=swiftshader')
options.add_argument('--disable-gpu-sandbox')
options.add_argument('--ignore-gpu-blocklist')

# Prevent Service Worker & UDP Leaks
options.add_argument('--disable-service-workers')
options.add_argument('--disable-features=ServiceWorker,UserAgentClientHint')
options.add_experimental_option("prefs", {
    "webrtc.ip_handling_policy": "disable_non_proxied_udp",
    "webrtc.multiple_routes_enabled": False,
    "webrtc.nonproxied_udp_enabled": False
})

# Launch physical window maximized to ensure stability
options.add_argument("--window-size=1920,1080")
options.add_argument('--window-position=0,0')

options.add_argument(f"--user-agent={sys_info['user_agent']}")
options.add_argument(f"--user-data-dir={user_data_dir}")

print("🚀 Launching Comprehensive Stealth Browser...")
driver = uc.Chrome(options=options, headless=False, version_main=int(sys_info['cv_major']))

try:
    # ====================================================================
    # --- 10. APPLY CDP OVERRIDES & HARDWARE FIXES ---
    # ====================================================================
    ua_metadata = {
        "brands": [
            {"brand": "Chromium", "version": sys_info['cv_major']},
            {"brand": "Google Chrome", "version": sys_info['cv_major']},
            {"brand": "Not-A.Brand", "version": "99"}
        ],
        "fullVersionList": [
            {"brand": "Chromium", "version": sys_info['cv_full']},
            {"brand": "Google Chrome", "version": sys_info['cv_full']},
            {"brand": "Not-A.Brand", "version": "99.0.0.0"}
        ],
        "platform": sys_info.get("ch_platform", "Linux"),
        "platformVersion": sys_info.get("ch_version", "5.15.0"),
        "architecture": sys_info.get("ch_arch", "x86"),
        "model": "SM-S901B" if sys_info['is_mobile'] else "",
        "mobile": sys_info["is_mobile"],
        "bitness": "64",
        "wow64": False
    }

    driver.execute_cdp_cmd('Emulation.setTimezoneOverride', {
        'timezoneId': sys_info['timezone']
    })

    driver.execute_cdp_cmd('Network.setUserAgentOverride', {
        "userAgent": sys_info['user_agent'],
        "platform": sys_info['platform'],
        "acceptLanguage": "en-US,en",
        "userAgentMetadata": ua_metadata
    })

    # CDP Hardware Touch and Mobile Viewport Emulation
    if sys_info['is_mobile']:
        driver.execute_cdp_cmd('Emulation.setDeviceMetricsOverride', {
            'width': sys_info['width'],
            'height': sys_info['height'],
            'deviceScaleFactor': 3,
            'mobile': True,
            'fitWindow': False
        })
        driver.execute_cdp_cmd('Emulation.setTouchEmulationEnabled', {
            'enabled': True,
            'maxTouchPoints': 5
        })
        driver.execute_cdp_cmd('Emulation.setEmitTouchEventsForMouse', {
            'enabled': True,
            'configuration': 'mobile'
        })

    # ====================================================================
    # --- 11. ADVANCED JAVASCRIPT PAYLOAD (Double-Escaped Curly Braces) ---
    # ====================================================================
    stealth_injection_js = f"""
    (function() {{
        const makeNative = (fn, name) => {{
            Object.defineProperty(fn, 'name', {{ value: name, configurable: true }});
            const str = "function " + name + "() {{ [native code] }}";
            fn.toString = () => str;
            fn.toString.toString = () => "function toString() {{ [native code] }}";
        }};

        // WebGL Spoofing (Main Window Context)
        ['WebGLRenderingContext', 'WebGL2RenderingContext'].forEach(function(ctx) {{
            if (window[ctx]) {{
                const originalGetParameter = window[ctx].prototype.getParameter;
                const spoofedGetParameter = function(parameter) {{
                    const result = originalGetParameter.apply(this, arguments);
                    if (parameter === 37445) return '{sys_info['webgl_vendor']}';
                    if (parameter === 37446) return '{sys_info['webgl_renderer']}';
                    return result;
                }};
                Object.defineProperty(window[ctx].prototype, 'getParameter', {{
                    value: spoofedGetParameter, writable: true, configurable: true, enumerable: false
                }});
                makeNative(spoofedGetParameter, 'getParameter');
            }}
        }});

        // Hardware Sensor Mocking (Cameras, Mics, Batteries) to pass deep heuristics
        if (navigator.mediaDevices) {{
            const mockDevices = [
                {{ kind: 'videoinput', deviceId: 'cam-1', label: '{sys_info['cam_name']}', groupId: 'g1' }},
                {{ kind: 'audioinput', deviceId: 'mic-1', label: 'Internal Microphone', groupId: 'g2' }},
                {{ kind: 'audiooutput', deviceId: 'spk-1', label: 'Internal Speaker', groupId: 'g2' }}
            ];
            if ({'true' if sys_info['is_mobile'] else 'false'}) {{
                mockDevices.push({{ kind: 'videoinput', deviceId: 'cam-2', label: 'Back Camera', groupId: 'g1' }});
            }}
            navigator.mediaDevices.enumerateDevices = () => Promise.resolve(mockDevices);
            makeNative(navigator.mediaDevices.enumerateDevices, 'enumerateDevices');
        }}

        if (navigator.getBattery) {{
            const mockBattery = {{
                charging: {sys_info['battery_charging']}, chargingTime: 0, dischargingTime: Infinity, level: {sys_info['battery_level']},
                addEventListener: () => {{}}, removeEventListener: () => {{}}, dispatchEvent: () => {{}}
            }};
            navigator.getBattery = () => Promise.resolve(mockBattery);
            makeNative(navigator.getBattery, 'getBattery');
        }}

        // Memory & CPU overrides on the prototype
        Object.defineProperty(Navigator.prototype, 'deviceMemory', {{ get: () => {sys_info['ram']}, configurable: true, enumerable: true }});
        Object.defineProperty(Navigator.prototype, 'hardwareConcurrency', {{ get: () => {sys_info['cores']}, configurable: true, enumerable: true }});

        // Helper to dynamically resolve relative worker paths to absolute URLs
        const resolveURL = (url) => {{
            try {{ return new URL(url, window.location.href).href; }}
            catch (e) {{ return url; }}
        }};

        // SHARED WORKER SCRIPT INJECTION
        const workerPayload = `
            if (self.WorkerNavigator) {{
                const proto = self.WorkerNavigator.prototype;

                Object.defineProperty(proto, 'userAgent', {{ get: () => '{sys_info['user_agent']}', configurable: true, enumerable: true }});
                Object.defineProperty(proto, 'platform', {{ get: () => '{sys_info['platform']}', configurable: true, enumerable: true }});
                Object.defineProperty(proto, 'hardwareConcurrency', {{ get: () => {sys_info['cores']}, configurable: true, enumerable: true }});
                Object.defineProperty(proto, 'deviceMemory', {{ get: () => {sys_info['ram']}, configurable: true, enumerable: true }});

                Object.defineProperty(proto, 'userAgentData', {{
                    get: () => ({{
                        brands: [
                            {{ brand: 'Chromium', version: '{sys_info['cv_major']}' }},
                            {{ brand: 'Google Chrome', version: '{sys_info['cv_major']}' }}
                        ],
                        mobile: {'true' if sys_info['is_mobile'] else 'false'},
                        platform: '{sys_info['ch_platform']}',
                        getHighEntropyValues: async () => ({{
                            architecture: '{sys_info['ch_arch']}',
                            bitness: '64',
                            model: '{'SM-S901B' if sys_info['is_mobile'] else ''}',
                            platform: '{sys_info['ch_platform']}',
                            platformVersion: '{sys_info['ch_version']}'
                        }})
                    }}),
                    configurable: true, enumerable: true
                }});
            }}

            ['WebGLRenderingContext', 'WebGL2RenderingContext'].forEach(function(ctxName) {{
                if (self[ctxName]) {{
                    const origGetParam = self[ctxName].prototype.getParameter;
                    self[ctxName].prototype.getParameter = function(param) {{
                        const res = origGetParam.apply(this, arguments);
                        if (param === 37445) return '{sys_info['webgl_vendor']}';
                        if (param === 37446) return '{sys_info['webgl_renderer']}';
                        return res;
                    }};
                }}
            }});

            if (self.OffscreenCanvas) {{
                const origGetContext = self.OffscreenCanvas.prototype.getContext;
                self.OffscreenCanvas.prototype.getContext = function(type, attributes) {{
                    const ctx = origGetContext.apply(this, arguments);
                    if (ctx && (type === 'webgl' || type === 'webgl2')) {{
                        const origGetParam = ctx.getParameter;
                        ctx.getParameter = function(param) {{
                            const res = origGetParam.apply(this, arguments);
                            if (param === 37445) return '{sys_info['webgl_vendor']}';
                            if (param === 37446) return '{sys_info['webgl_renderer']}';
                            return res;
                        }};
                    }}
                    return ctx;
                }};
            }}
        `;

        // Intercept dedicated Workers
        const OriginalWorker = window.Worker;
        window.Worker = function(scriptURL, options) {{
            const resolved = resolveURL(scriptURL);
            const originUrl = window.location.href;
            const originHost = window.location.host;
            const originHostname = window.location.hostname;

            const locPayload = "Object.defineProperty(self, 'location', {{ get: () => ({{ href: '" + originUrl + "', protocol: 'https:', host: '" + originHost + "', hostname: '" + originHostname + "', pathname: '/', search: '', hash: '' }}), configurable: true }});";

            const rawScript = locPayload + "\\n" + workerPayload + "\\nimportScripts('" + resolved + "');";
            const blob = new Blob([rawScript], {{ type: 'application/javascript' }});
            return new OriginalWorker(URL.createObjectURL(blob), options);
        }};
        makeNative(window.Worker, 'Worker');

        // Intercept Shared Workers
        const OriginalSharedWorker = window.SharedWorker;
        window.SharedWorker = function(scriptURL, options) {{
            const resolved = resolveURL(scriptURL);
            const originUrl = window.location.href;
            const originHost = window.location.host;
            const originHostname = window.location.hostname;

            const locPayload = "Object.defineProperty(self, 'location', {{ get: () => ({{ href: '" + originUrl + "', protocol: 'https:', host: '" + originHost + "', hostname: '" + originHostname + "', pathname: '/', search: '', hash: '' }}), configurable: true }});";

            const rawScript = locPayload + "\\n" + workerPayload + "\\nimportScripts('" + resolved + "');";
            const blob = new Blob([rawScript], {{ type: 'application/javascript' }});
            return new OriginalSharedWorker(URL.createObjectURL(blob), options);
        }};
        makeNative(window.SharedWorker, 'SharedWorker');
    }})();
    """
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {'source': stealth_injection_js})

    # ====================================================================
    # --- 12. DYNAMIC TASK DISPATCHING & SCRAPING LOGIC ---
    # ====================================================================
    print("\n" + "="*50)
    print("🕵️ BEGINNING AUTONOMOUS SCRAPING LOOP")
    print("="*50)

    tailnet_proxies = {'http': 'socks5h://127.0.0.1:1055', 'https': 'socks5h://127.0.0.1:1055'}

    while True:
        # 1. ASK COLAB A FOR A TASK
        print(f"\n📡 Requesting next task from Master Node ({COLAB_A_IP})...")
        try:
            task_resp = requests.get(f"http://{COLAB_A_IP}:5000/get_task", proxies=tailnet_proxies, timeout=10)
            task_data = task_resp.json()

            if task_data['status'] == "done":
                print("🎉 Master node says all tasks are complete! Shutting down.")
                break

            target_url = task_data['url']
            print(f"   ➜ Task received! Scraping: {target_url}")

        except Exception as e:
            print(f"❌ Failed to reach Master Node. Ensure Colab A is running. Error: {e}")
            break

        # 2. PERFORM THE SCRAPING (Protected by Exit Node stealth)
        try:
            driver.get(target_url)
            time.sleep(2) # Emulate human read time

            # Extract basic data (You can customize this block for your specific targets)
            try:
                page_title = driver.title
            except:
                page_title = "Unknown Title"

            scraped_payload = {
                "scraper_node": f"Colab_Worker_{sys_info['platform']}_{sys_info['ch_platform']}",
                "target_url": target_url,
                "item_title": page_title,
                "timestamp": time.time()
            }

            # 3. SEND DATA BACK TO COLAB A
            response = requests.post(
                f"http://{COLAB_A_IP}:5000/data",
                json=scraped_payload,
                proxies=tailnet_proxies,
                timeout=10
            )
            print(f"   ➜ ✅ Successfully extracted and transmitted: {page_title}")

        except Exception as e:
            print(f"   ➜ ❌ Scraping failed for {target_url}: {e}")

finally:
    driver.quit()

🔒 Locking Tailscale routing strictly to Exit Node: 100.86.149.127...
🧹 Cleaning persistent Service Worker cache...
✅ Dynamic Engine detected Exit Node: OS='MACOS' | ARCH=''

📡 Checking SOCKS5 connectivity & fetching dynamic exit node geolocation...
✅ SOCKS5 proxy is healthy!
🌍 Exit Node Location: Gangtok, India (Timezone: Asia/Kolkata)

🖥️ Starting Xvfb Virtual Display (1920x1080)...
🚀 Launching Comprehensive Stealth Browser...

🕵️ BEGINNING AUTONOMOUS SCRAPING LOOP

📡 Requesting next task from Master Node (100.96.27.123)...
   ➜ Task received! Scraping: https://books.toscrape.com/catalogue/page-50.html
   ➜ ✅ Successfully extracted and transmitted: All products | Books to Scrape - Sandbox

📡 Requesting next task from Master Node (100.96.27.123)...
   ➜ Task received! Scraping: https://books.toscrape.com/catalogue/page-50.html
   ➜ ✅ Successfully extracted and transmitted: All products | Books to Scrape - Sandbox

📡 Requesting next task from Master Node (100.96.27.123)...
🎉 Master node